In [1]:
# Bật GPU (T4/P100) và Internet trong Settings trước khi chạy
!git clone https://github.com/hotuyen21pt/MultiLABSA.git

# Kaggle đã có sẵn torch/transformers/pandas/numpy/tqdm — chỉ cài thứ còn thiếu,
# KHÔNG cài lại torch để tránh phá bản CUDA của Kaggle.
!pip install -q sentencepiece protobuf
# Thêm (chỉ cần nếu chạy build_lexicon.py):
!pip install -q wordfreq jieba fasttext-wheel

Cloning into 'MultiLABSA'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 244 (delta 61), reused 237 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (244/244), 7.10 MiB | 15.05 MiB/s, done.
Resolving deltas: 100% (61/61), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 10.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 14.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 9.4 MB/s eta 0:00:00


In [ ]:
import os, glob, torch

candidates = glob.glob("/kaggle/input/**/unlabeled_data", recursive=True)
if not candidates:
    print("Cac dataset dang co duoi /kaggle/input:")
    for name in os.listdir("/kaggle/input"):
        print(" -", name)
    raise FileNotFoundError(
        "Khong tim thay thu muc 'unlabeled_data' duoi /kaggle/input. "
        "Kiem tra da them dataset trong Settings > Add Input, roi xem danh sach in o tren."
    )
DATA = candidates[0]
print("DATA =", DATA)

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print("Files:", os.listdir(DATA))   # kỳ vọng: hotel_review1_lang.csv, ...

In [ ]:
%cd /kaggle/working/MultiLABSA/dapt
!python build_lexicon.py \
    --data_dir {DATA} \
    --languages en vi fr de es it nl ru ja ko zh \
    --max_per_lang 200000 --top_k 300 --top_k_bigrams 100 \
    --min_general_freq 1e-6 --expand \
    --out_dir /kaggle/working/lexicon

In [ ]:
%cd /kaggle/working/MultiLABSA/dapt
!python train_dapt.py \
    --data_dir {DATA} \
    --lexicon_file /kaggle/working/lexicon/lexicon.json \
    --precision auto \
    --batch_size 8 --gradient_accumulation_steps 4 \
    --num_epochs 1 --max_steps 5000 \
    --warmup_steps 500 --eval_steps 500 --save_steps 500 \
    --output_dir /kaggle/working/checkpoints \
    --final_dir /kaggle/working/hotel-mt5

In [ ]:
%cd /kaggle/working/MultiLABSA/dapt
!python train_dapt.py --resume \
    --data_dir {DATA} \
    --lexicon_file /kaggle/working/lexicon/lexicon.json \
    --output_dir /kaggle/working/checkpoints \
    --final_dir /kaggle/working/hotel-mt5

In [6]:
import shutil
shutil.make_archive("/kaggle/working/hotel-mt5", "zip", "/kaggle/working/hotel-mt5")
print("Saved /kaggle/working/hotel-mt5.zip")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/hotel-mt5'